# 21장 · 갈림길의 정원

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [21장 · 갈림길의 정원](https://grow.minds.kr/textbooks/css-methods/causal/book/ch21-갈림길의-정원.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. 갈림길의 정원
효과가 확실히 없는 세계(집단 표시를 뒤섞음)에서, 분석 갈림길을 다 걸으면 '유의'가 수확됩니다.

In [ ]:
exp = pd.read_csv("data/journey_exp.csv")
rng = np.random.default_rng(73)
null_cond = rng.permutation(exp.cond.values)
d = exp.assign(cond=null_cond); dc = d[d.attn_1==1]
tt = stats.ttest_ind(dc[dc.cond==1].mil, dc[dc.cond==0].mil)
print("사전 지정 주 분석 p =", round(tt.pvalue,3))   # 0.346 (정직한 답)

## 3. 60갈래를 다 걸으면
결과 2 × 표본 3 × 하위집단 5 × 공변인 2 = 60. 하나하나는 합리적인데, 다 걸어 예쁜 것만 고르면?

In [ ]:
def forks(df, cond):
    out=[]; d=df.copy(); d["cond"]=cond
    for oc in ("mil","flr"):
        for excl in ("none","attn","trim"):
            e=d
            if excl=="attn": e=d[d.attn_1==1]
            elif excl=="trim":
                z=(d[oc]-d[oc].mean())/d[oc].std(ddof=1); e=d[np.abs(z)<=2.5]
            for sub in ("all","m","f","young","old"):
                s=e
                if sub=="m": s=e[e.gender==1]
                elif sub=="f": s=e[e.gender==2]
                elif sub=="young": s=e[e.age<=e.age.median()]
                elif sub=="old": s=e[e.age>e.age.median()]
                for cov in (False, True):        # 공변인(기저 mil_t1) 넣기/빼기
                    if cov:
                        _,_,p,_ = ols(s[oc].values, [s.cond.values, s.mil_t1.values]); out.append(p[1])
                    else:
                        out.append(stats.ttest_ind(s[s.cond==1][oc], s[s.cond==0][oc]).pvalue)
    return out
ps = forks(exp, null_cond)
print("갈림길:", len(ps), " p<.05 수확:", sum(p<.05 for p in ps), " 최소 p:", round(min(ps),3))
# 사전 지정 하나(0.346)는 정직하지만, 60을 다 걸으면 그럴듯한 유의가 여럿 나온다.

## 4. 탐색: AIC 전진 선택 · 떼어 둔 반 · 5겹 교차 검증 (책 §21.2 · 복습 ⑧)
잡음 세계(스무 후보 가운데 결과와 관련된 변수가 없다)와 신호 세계(두 변수가 결과를 정한다)를 200개씩 만듭니다. 출력 두 줄은 세계별 평균이고, 순서는 고른 예측변수 수 · 그 가운데 참 예측변수가 아닌 것 · 그 가운데 *p* < .05인 것 · 탐색 반 *R*² · 떼어 둔 반 *R*² · 떼어 둔 반 *R*²가 0보다 작은 세계의 비율 · 5겹 교차 검증 *R*²입니다. 앞 상자와 이어지지 않으므로 따로 돌려도 됩니다. ⏳ 1분쯤 걸립니다.

In [ ]:
def aic(y, cols):
    X1 = np.column_stack([np.ones(len(y))] + cols)
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    rss = ((y - X1 @ b) ** 2).sum()
    return len(y) * np.log(rss / len(y)) + 2 * X1.shape[1]


def 전진선택(X, y):
    고른 = []
    지금 = aic(y, [])
    while True:
        후보 = [j for j in range(X.shape[1]) if j not in 고른]
        if not 후보:
            break
        점수 = [aic(y, [X[:, k] for k in 고른 + [j]]) for j in 후보]
        if min(점수) >= 지금:                    # 더 줄일 후보가 없으면 멈춘다
            break
        지금 = min(점수)
        고른.append(후보[int(np.argmin(점수))])
    return 고른


def 예측R2(y, 예측):
    return 1 - ((y - 예측) ** 2).sum() / ((y - y.mean()) ** 2).sum()


def 고르고_예측(X, y, 맞출, 볼):
    고른 = 전진선택(X[맞출], y[맞출])
    b, _, p, r2 = ols(y[맞출], [X[맞출][:, k] for k in 고른])
    return 고른, p, r2, b[0] + X[볼][:, 고른] @ b[1:]


def 한_세계(X, y, 참):
    탐색, 확인 = np.arange(100), np.arange(100, 200)          # 앞 100명 · 뒤 100명
    고른, p, r2, 예측 = 고르고_예측(X, y, 탐색, 확인)
    떼어둔 = 예측R2(y[확인], 예측)
    cv예측 = np.empty(200)
    for f in range(5):                                        # 40명씩 다섯 몫
        볼 = np.arange(40 * f, 40 * f + 40)
        맞출 = np.setdiff1d(np.arange(200), 볼)
        cv예측[볼] = 고르고_예측(X, y, 맞출, 볼)[3]             # 선택부터 다시 한다
    return [len(고른), sum(k not in 참 for k in 고른), (p[1:] < .05).sum(),
            r2, 떼어둔, 떼어둔 < 0, 예측R2(y, cv예측)]


rng = np.random.default_rng(73)
잡음, 신호 = [], []
for _ in range(200):
    X = rng.standard_normal((200, 20))                        # 후보 예측변수 20개
    e = rng.standard_normal(200)
    잡음.append(한_세계(X, e, []))
    신호.append(한_세계(X, 0.5 * X[:, 0] + 0.5 * X[:, 1] + e, [0, 1]))
for 이름, 결과 in (("잡음", 잡음), ("신호", 신호)):
    m = np.mean(결과, axis=0)
    print(이름, *[round(float(v), d) for v, d in zip(m, (1, 1, 1, 2, 2, 2, 2))])
# 잡음 3.6 3.6 1.5 0.12 -0.17 0.97 -0.09
# 신호 5.2 3.2 3.2 0.42 0.21 0.05 0.27

## 5. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.